# Brazilian E-Commerce Analysis

## Data Cleaning & Preparation

In this notebook we will:

- Load data from SQL Server
- Inspect the dataset
- basic python poerations on the dataset 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sqlalchemy import create_engine
import urllib
from pathlib import Path

In [2]:
# SQL Server Connection

server = r"localhost\SQLEXPRESS"
database = "Brazilian E_Commerce"

connection_string = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={connection_string}"
)

In [3]:
# Read SQL Query

sql_file = (
    Path.cwd().parent
    / "SQL"
    / "reports"
    / "Executive_dashboard_dataset.sql"
)

with open(sql_file, "r", encoding="utf-8") as file:
    query = file.read()

In [4]:
df = pd.read_sql(query, engine)

In [5]:
df.head()

,order_date,order_delivered_customer_date,customer_id,customer_city,customer_state,product_id,product_category_name,seller_id,quantity,payment_value,freight_value,revenue
0,2017-09-13 08:59:02,2017-09-20 23:43:48,3ce436f183e68e07877b285a838db11a,campos dos goytacazes,RJ,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,48436dade18ac8b2bce089ec2a041202,1,72.19,13.29,58.90
1,2017-04-26 10:53:06,2017-05-12 16:04:24,f6dd3ec061db4e3987629fe6b26e5cce,santa fe do sul,SP,e5f2d52b802189ee658865ca93d83a8f,pet_shop,dd7ddc04e1b6c2c614352b383efe2d36,1,259.83,19.93,239.90
2,2018-01-14 14:33:31,2018-01-22 13:19:16,6489ae5e4333f3693df5ad4372dab6d3,para de minas,MG,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,5b51032eddd242adc84c38acab88f23d,1,216.87,17.87,199.00
3,2018-08-08 10:00:35,2018-08-14 13:32:39,d4eb9395c8c0431ee92fce09860c5a06,atibaia,SP,7634da152a4610f1595efa32f14722fc,perfumaria,9d7a1d34a5052409006425275ba1c2b4,1,25.78,12.79,12.99
4,2017-02-04 13:57:51,2017-03-01 16:42:31,58dbd0b2d70206bf40e62cd34e84d795,varzea paulista,SP,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,df560393f3a51e74553ab94004ba5c87,1,218.04,18.14,199.90


Create a function to calculate the total order value

In [6]:
def calculate_total(price, freight, quantity):
    return (price * quantity) + freight

In [7]:
df["total_order_value"] = df.apply(
    lambda x: calculate_total(
        x["revenue"],
        x["freight_value"],
        x["quantity"]
    ),
    axis=1
)

df[["revenue","quantity","freight_value","total_order_value"]].head()

,revenue,quantity,freight_value,total_order_value
0,58.90,1,13.29,72.19
1,239.90,1,19.93,259.83
2,199.00,1,17.87,216.87
3,12.99,1,12.79,25.78
4,199.90,1,18.14,218.04


Classify Orders

In [8]:
def classify_order(payment):

    if payment >= 500:
        return "Premium"

    elif payment >= 100:
        return "Standard"

    else:
        return "Low Value"

In [9]:
df["order_type"] = df["payment_value"].apply(classify_order)

df[["payment_value","order_type"]].head()

,payment_value,order_type
0,72.19,Low Value
1,259.83,Standard
2,216.87,Standard
3,25.78,Low Value
4,218.04,Standard


Loop through payment values

In [10]:
above_500 = 0
below_100 = 0

for payment in df["payment_value"]:

    if payment > 500:
        above_500 += 1

    elif payment < 100:
        below_100 += 1

print("Orders Above 500:", above_500)
print("Orders Below 100:", below_100)

Orders Above 500: 6138
Orders Below 100: 54157


Average Product Price For Each Category

In [11]:
category_avg = {}

for category in df["product_category_name"].dropna().unique():

    avg_price = df[df["product_category_name"] == category]["revenue"].mean()

    category_avg[category] = avg_price

In [12]:
most_expensive = max(category_avg, key=category_avg.get)

print("Most Expensive Category:", most_expensive)
print("Average Price:", category_avg[most_expensive])

Most Expensive Category: pcs
Average Price: 1103.6891363636364


Delivery Time

In [6]:
df["order_date"] = pd.to_datetime(df["order_date"])

df["order_delivered_customer_date"] = pd.to_datetime(
    df["order_delivered_customer_date"]
)

In [11]:
def calculate_delivery_duration(purchase_date, delivered_date):
    return (delivered_date - purchase_date).days

In [12]:
df["delivery_duration"] = df.apply(
    lambda row: calculate_delivery_duration(
        row["order_date"],
        row["order_delivered_customer_date"]
    ),
    axis=1
)

df[
    [
        "order_date",
        "order_delivered_customer_date",
        "delivery_duration"
    ]
].head()

,order_date,order_delivered_customer_date,delivery_duration
0,2017-09-13 08:59:02,2017-09-20 23:43:48,7.0
1,2017-04-26 10:53:06,2017-05-12 16:04:24,16.0
2,2018-01-14 14:33:31,2018-01-22 13:19:16,7.0
3,2018-08-08 10:00:35,2018-08-14 13:32:39,6.0
4,2017-02-04 13:57:51,2017-03-01 16:42:31,25.0
